# 5장 · 기술과 시각화: 표 1

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 설치는 없고, 구글 계정만 있으면 됩니다.

**읽는 법.** 지금 읽고 있는 이런 글은 설명이라 실행할 것이 없습니다. 코드는 **왼쪽에 `[ ]` 와 ▶ 단추가 붙은 상자**이고, 그 ▶ 또는 `Shift`+`Enter` 로 실행합니다.

**순서.** 번호 차례대로 끝까지 갑니다. **1 준비**부터 시작해 마지막 번호까지 위에서 아래로 내려가면 됩니다. ⚠ 가운데부터 누르면 앞에서 만든 것이 없어 오류가 납니다.

📖 본문 학습 페이지: [5장 · 기술과 시각화: 표 1](https://grow.minds.kr/textbooks/css-methods/causal/book/ch05-기술과-시각화.html)

## 1. 준비

아래 상자 **둘**을 차례로 실행하세요. 둘 다 해야 그다음이 돌아갑니다.

1. 첫째 = 자료와 코드를 내려받습니다. 몇 초 걸리고 「경고」 문구가 떠도 정상입니다. 상자 아래에 `/content/css-methods-causal-code` 가 찍히면 성공입니다.
2. 둘째 = 도구를 불러옵니다. **준비 끝**이라는 줄이 찍히면 됩니다.

⚠ 한참 뒤에 돌아와 `pd`·`exp`가 없다거나 파일이 없다는 오류가 나면 코랩이 연결을 끊은 것입니다. 자료는 내 컴퓨터가 아니라 코랩의 컴퓨터에 있었으므로 함께 사라졌습니다. **이 두 상자를 위에서부터 다시** 누르면 됩니다(첫째 상자는 여러 번 눌러도 됩니다).

> **어느 것이 상자인가.** 왼쪽에 `[ ]` 표시와 **▶** 단추가 붙은 것이 코드 상자입니다. 글씨에 색이 섞여 있습니다. 지금 읽고 있는 이런 글에는 `[ ]` 도 ▶ 도 없습니다.

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(수 초). 다시 눌러도 됩니다.
%cd /content
![ -d css-methods-causal-code ] || git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd /content/css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


💡 **방금 둘째 상자에 코드가 길게 펼쳐졌는데, 지금 이해하지 않아도 됩니다.**

이 책 전체가 쓰는 **도우미 넷**을 미리 만들어 두는 곳이라 그렇습니다. 지금은 이름과 하는 일만 훑고 넘어가세요. 안에 든 식은 필요한 장에서 하나씩 만납니다.

| 이름 | 하는 일 | 처음 쓰는 곳 |
|---|---|---|
| `load` | 자료 한 벌을 불러오고 주의 점검 실패자를 걸러 낸다 | 2장 |
| `cronbach` | 척도 신뢰도(알파) | 4장 |
| `cohen_d` | 두 집단 차이의 효과크기 | 13장 |
| `ols` | 회귀의 계수·표준오차·p·R² | 14장 |

⚠ `np.linalg.lstsq` 처럼 낯선 이름이 보여도 괜찮습니다. **그것이 이 책이 「계산은 AI가」라고 말하는 곳입니다.**

⭐ **그리고 이 넷은 여러분이 검증할 대상이 아닙니다.** 이 책이 검증해 둔 도구입니다(배포 때마다 대조 배터리가 이 함수들의 출력을 본문 수치와 맞춰 봅니다). 여러분이 읽어야 할 코드는 **AI가 준 코드**이고, 그 훈련은 3장부터 시작합니다.

📖 이 넷을 다시 보고 싶으면 책 부록 E.1b 에 같은 코드가 있습니다.

## 2. 표 1 만들기 (실습 1)
논문의 첫 표. 주요 변수의 평균 · 표준편차 · 관측 최소 · 최대를 한 표로 찍습니다. 책 §5.5 표 1의 「관측 범위」 열과 맞춰 보세요. 「가능 범위」(1 ～ 7)는 코드가 아니라 척도의 설계에서 옵니다.

In [ ]:
svy = load("svy")
cols = ["age", "hjs", "mil", "flr", "swl"]
print(svy[cols].agg(["mean", "std", "min", "max"]).T.round(2))
#       mean    std    min    max
# age  45.87  12.37  18.00  75.00
# hjs   5.01   0.80   2.33   6.86
# mil   4.89   1.25   1.00   7.00
# flr   5.23   0.88   2.12   7.00
# swl   4.60   1.10   1.20   7.00

## 3. 상관: 이 책 내내 따라올 숫자
여정과 의미의 상관 .63. 아직 아무 인과도 아닙니다(6·7장).

In [ ]:
print("r(여정, 의미) =", round(svy.hjs.corr(svy.mil), 2))

## 4. 분포를 그림으로 (실습 2 · 3 의 출발점)
§5.1 의 히스토그램과 평균선, §5.3 의 성별 바이올린입니다. ⚠ 한글 축 이름이 네모로 나오면 글꼴을 지정하세요(부록 E 의 `plt.rcParams["font.family"]` 안내).

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(svy["mil"], bins=20)                       # 점수대별 사람 수
ax.axvline(svy["mil"].mean(), linestyle="--")      # 평균선
ax.set_xlabel("삶의 의미 (1~7점)")
ax.set_ylabel("사람 수")

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.violinplot([svy[svy.gender == 1]["mil"], svy[svy.gender == 2]["mil"]])   # 1 = 남, 2 = 여
ax.set_xticks([1, 2], ["남", "여"])

## 5. 참여자 문단에 옮길 수 (실습 5)
표 1 주와 참여자 문단에 들어갈 인원 · 나이 · 성별 비율을 코드 출력에서 바로 찍습니다(§5.6). 손으로 옮기지 않습니다.

In [ ]:
raw = load("svy", clean=False)
print(len(raw), len(raw) - len(svy), len(svy))          # 응답 · 뺀 사람 · 분석
print(round(svy.age.mean(), 1), round(svy.age.std(), 1), svy.age.min(), svy.age.max())
n = svy.gender.value_counts().sort_index()
print(n.tolist(), (n / len(svy) * 100).round(1).tolist())
# 590 24 566
# 45.9 12.4 18 75
# [270, 284, 12] [47.7, 50.2, 2.1]

## 6. 직접 바꿔 보기
위 셀의 숫자(씨앗 73, 표본 크기, 제외 기준 등)를 바꿔 다시 실행해 보세요. 결과가 어떻게 달라지나요?

> **검증 로그(부록 B)**: 무엇을 바꿨고, 무엇이 나왔고, 예상과 같았는지 한 문단으로 적어 두세요. 실행이 아니라 검증이 이 책의 핵심입니다.